# Unsloth Verilog Smoke Test

**Goal**: Prove the gcc/Triton/Unsloth training path works end-to-end on a tiny Verilog dataset.

**Model**: `unsloth/Qwen3.5-0.8B`

**Run size**: 3 training steps on a tiny in-memory dataset

**Cleanup**: Final cell removes smoke-test outputs and caches


In [8]:
# Configure compiler/toolchain and install training packages
import os
import shutil
import subprocess
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore', message='.*_check_is_size.*', category=FutureWarning)
os.environ.setdefault('TRITON_CACHE_DIR', '/tmp/triton-cache')
os.environ.setdefault('TORCHINDUCTOR_CACHE_DIR', '/tmp/torchinductor-cache')

compiler_candidates = [
    ('/tmp/conda-envs/gcc_env/bin/gcc', '/tmp/conda-envs/gcc_env/bin/g++'),
    ('/tmp/conda-envs/gcc_env/bin/x86_64-conda-linux-gnu-gcc', '/tmp/conda-envs/gcc_env/bin/x86_64-conda-linux-gnu-g++'),
    (shutil.which('gcc'), shutil.which('g++')),
    (shutil.which('x86_64-conda-linux-gnu-gcc'), shutil.which('x86_64-conda-linux-gnu-g++')),
]

selected_cc = None
selected_cxx = None
for cc, cxx in compiler_candidates:
    if cc and cxx and Path(cc).exists() and Path(cxx).exists():
        selected_cc = str(Path(cc).resolve())
        selected_cxx = str(Path(cxx).resolve())
        break

SMOKE_COMPILER_READY = selected_cc is not None and selected_cxx is not None
SMOKE_SELECTED_CC = selected_cc
SMOKE_SELECTED_CXX = selected_cxx

if SMOKE_COMPILER_READY:
    compiler_bin = str(Path(selected_cc).parent)
    os.environ['PATH'] = compiler_bin + os.pathsep + os.environ.get('PATH', '')
    os.environ['CC'] = selected_cc
    os.environ['CXX'] = selected_cxx
    print(f"Using C compiler: {os.environ['CC']}")
    print(f"Using C++ compiler: {os.environ['CXX']}")
    print(subprocess.check_output([os.environ['CC'], '--version'], text=True).splitlines()[0])
else:
    os.environ['TORCH_COMPILE'] = '0'
    os.environ['UNSLOTH_COMPILE_DISABLE'] = '1'
    os.environ['TRITON_COMPILE_DISABLE'] = '1'
    os.environ['VLLM_DISABLE_COMPILE_CACHE'] = '1'
    print('No compiler detected; using compile-disabled fallback mode.')

print('Installing Unsloth stack...')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--user', '-q', 'unsloth', 'trl', 'peft', 'accelerate'])
print('Setup complete.')


Using C compiler: /usr/local/bin/x86_64-conda-linux-gnu-gcc
Using C++ compiler: /usr/local/bin/x86_64-conda-linux-gnu-g++
x86_64-conda-linux-gnu-gcc (Anaconda gcc) 11.2.0
Installing Unsloth stack...
Setup complete.


In [9]:
# Verify compiler and GPU state
import os
import subprocess
import torch

print(f"Compiler ready: {SMOKE_COMPILER_READY}")
print(f"CC: {os.environ.get('CC', 'unset')}")
print(f"CXX: {os.environ.get('CXX', 'unset')}")
if SMOKE_COMPILER_READY:
    print(subprocess.check_output([os.environ['CC'], '--version'], text=True).splitlines()[0])
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")


Compiler ready: True
CC: /usr/local/bin/x86_64-conda-linux-gnu-gcc
CXX: /usr/local/bin/x86_64-conda-linux-gnu-g++
x86_64-conda-linux-gnu-gcc (Anaconda gcc) 11.2.0
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
VRAM: 14.6 GB


In [10]:
# Build a tiny in-memory Verilog dataset
from datasets import Dataset

tiny_rows = [
    {"instruction": "Create a 2-input AND gate.", "output": "module and2(input a, input b, output y); assign y = a & b; endmodule"},
    {"instruction": "Create a 2-input OR gate.", "output": "module or2(input a, input b, output y); assign y = a | b; endmodule"},
    {"instruction": "Create a 2-input XOR gate.", "output": "module xor2(input a, input b, output y); assign y = a ^ b; endmodule"},
    {"instruction": "Create an inverter.", "output": "module inv1(input a, output y); assign y = ~a; endmodule"},
    {"instruction": "Create a 2-to-1 mux.", "output": "module mux2(input a, input b, input s, output y); assign y = s ? b : a; endmodule"},
    {"instruction": "Create an 8-bit register with synchronous reset.", "output": "module reg8(input clk, input rst, input [7:0] d, output reg [7:0] q); always @(posedge clk) begin if (rst) q <= 8\'b0; else q <= d; end endmodule"},
]

def format_row(row):
    return {
        "text": (
            f"### Instruction:\nGenerate Verilog code for: {row['instruction']}\n\n"
            f"### Response:\n{row['output']}"
        )
    }

tiny_dataset = Dataset.from_list(tiny_rows).map(format_row, remove_columns=["instruction", "output"])
print(f"Tiny dataset size: {len(tiny_dataset)}")
print(tiny_dataset[0]["text"])


Map:   0%|          | 0/6 [00:00<?, ? examples/s]

Tiny dataset size: 6
### Instruction:
Generate Verilog code for: Create a 2-input AND gate.

### Response:
module and2(input a, input b, output y); assign y = a & b; endmodule


In [11]:
# Load a very small model for smoke testing
import torch
from unsloth import FastLanguageModel

if not globals().get("SMOKE_COMPILER_READY", False):
    torch.compiler.disable(recursive=True)
    print("Compiler unavailable: using compile-disabled fallback for smoke test.")
else:
    print(f"Compiler available for Triton builds: {os.environ.get('CC')}")

max_seq_length = 256
load_in_4bit = True
model_name = "unsloth/Qwen3.5-0.8B"

print(f"Loading smoke-test model: {model_name}")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    load_in_4bit=load_in_4bit,
    load_in_16bit=False,
    full_finetuning=False,
)

print("Model load complete.")


Compiler available for Triton builds: /usr/local/bin/x86_64-conda-linux-gnu-gcc
Loading smoke-test model: unsloth/Qwen3.5-0.8B
==((====))==  Unsloth 2026.3.11: Fast Qwen3_5 patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model.safetensors-00001-of-00001.safeten(…):   0%|          | 0.00/1.75G [00:00<?, ?B/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

...

preprocessor_config.json:   0%|          | 0.00/336 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/20.0M [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

Model load complete.


In [12]:
# Configure LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=4,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=8,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    max_seq_length=max_seq_length,
)

print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


Unsloth: Making `model.base_model.model.model.language_model` require gradients
Trainable parameters: 1,597,440


In [13]:
# Configure a tiny trainer run
from trl import SFTConfig, SFTTrainer

smoke_output_dir = "outputs_verilog_llm_smoke"
max_steps = 3

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=tiny_dataset,
    dataset_text_field="text",
    args=SFTConfig(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        max_steps=max_steps,
        learning_rate=2e-5,
        lr_scheduler_type="cosine",
        warmup_steps=1,
        optim="adamw_8bit",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        output_dir=smoke_output_dir,
        save_strategy="no",
        seed=3407,
        dataset_num_proc=1,
        report_to=[],
    ),
)

print(f"Smoke trainer ready for {max_steps} steps.")


Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"] (num_proc=1):   0%|          | 0/6 [00:00<?, ? examples/s]

Smoke trainer ready for 3 steps.


In [14]:
# Run the smoke-test training loop
import time

start_time = time.time()
trainer_stats = trainer.train()
elapsed = time.time() - start_time

print("Smoke test training finished.")
print(f"Elapsed seconds: {elapsed:.1f}")
print(f"Train loss: {trainer_stats.training_loss}")


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 6 | Num Epochs = 1 | Total steps = 3
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 1 x 1) = 1
 "-____-"     Trainable parameters = 1,597,440 of 854,583,360 (0.19% trained)


Step,Training Loss
1,2.520177
2,2.493505
3,1.909113


Smoke test training finished.
Elapsed seconds: 56.8
Train loss: 2.3075981934865317


In [15]:
# Cleanup smoke-test outputs and caches
import gc
import shutil
from pathlib import Path

cleanup_paths = [
    Path("outputs_verilog_llm_smoke"),
    Path("/tmp/triton-cache"),
    Path("/tmp/torchinductor-cache"),
    Path.cwd() / "unsloth_compiled_cache",
]

for obj_name in ["trainer", "model", "tokenizer"]:
    if obj_name in globals():
        del globals()[obj_name]

gc.collect()

try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except Exception as cleanup_error:
    print(f"CUDA cleanup warning: {cleanup_error}")

for path in cleanup_paths:
    if path.exists():
        if path.is_dir():
            shutil.rmtree(path)
        else:
            path.unlink()
        print(f"Removed: {path}")
    else:
        print(f"Not present: {path}")

print("Smoke-test cleanup complete.")


Removed: outputs_verilog_llm_smoke
Removed: /tmp/triton-cache
Removed: /tmp/torchinductor-cache
Removed: /home/jovyan/silicogen/rtl_analyzer/notebooks/rtl_analyzer_phase3/unsloth_compiled_cache
Smoke-test cleanup complete.
